# TokenSage — Hybrid Token-Efficient Routing Agent

**AMD Developer Hackathon ACT II — Track 1**

Routes tasks between a local vLLM model (free) and a larger model (Fireworks or local) to minimize Fireworks token spend while maintaining accuracy.

### Architecture
- **Router**: Task-type classification + few-shot LLM routing with confidence
- **Local Solver**: vLLM offline (Qwen2.5-3B), structured JSON output
- **Big Solver**: Fireworks AI (72B) if API key present, else vLLM larger model (14B)
- **Verifier**: Re-answer from scratch (not "is this correct?") — catches local mistakes
- **Retry**: Local retry with chain-of-thought before escalating
- **Token Accounting**: Exact Fireworks token counts logged per-task

In [ ]:
# ============================================================
# Cell 1: Environment Check & Dependencies
# ============================================================
import subprocess, sys, os

# Check ROCm
try:
    import torch
    print(f"ROCm available: {torch.cuda.is_available()}")
    print(f"HIP version: {getattr(torch.version, 'hip', None)}")
    print(f"GPU count: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"torch not found or error: {e}")

# Ensure critical packages are installed
required = ['vllm', 'openai', 'pydantic']
for pkg in required:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: installing...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print("\nEnvironment ready.")

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os, json, sys, time, re
from typing import Optional, Any
from openai import OpenAI
from vllm import LLM, SamplingParams
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Environment Variables ---
# LOCAL_MODEL: HF model ID for the local small model (default Qwen2.5-3B-Instruct)
LOCAL_MODEL = os.getenv("LOCAL_MODEL", "Qwen/Qwen2.5-3B-Instruct")

# BIG_MODEL_LOCAL: HF model ID for fallback when no Fireworks key (default Qwen2.5-14B-Instruct)
BIG_MODEL_LOCAL = os.getenv("BIG_MODEL", "Qwen/Qwen2.5-14B-Instruct")

# Fireworks configuration
FIREWORKS_API_KEY = os.getenv("FIREWORKS_API_KEY", "")
FIREWORKS_BASE_URL = os.getenv("FIREWORKS_BASE_URL", "https://api.fireworks.ai/inference/v1")
FIREWORKS_MODEL = os.getenv("FIREWORKS_MODEL", "accounts/fireworks/models/qwen2p5-72b-instruct")

# Mistral configuration (fallback if Fireworks is unavailable)
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")
MISTRAL_BASE_URL = os.getenv("MISTRAL_BASE_URL", "https://api.mistral.ai/v1")
MISTRAL_MODEL = os.getenv("MISTRAL_MODEL", "mistral-large-latest")

# Determine providers
USE_FIREWORKS = bool(FIREWORKS_API_KEY)
USE_MISTRAL = bool(MISTRAL_API_KEY) and not USE_FIREWORKS

# Input/Output paths (for scoring submission)
INPUT_PATH = os.getenv("INPUT_PATH", "/input/tasks.json")
OUTPUT_PATH = os.getenv("OUTPUT_PATH", "/output/results.json")

# Thresholds
ROUTER_CONFIDENCE_THRESHOLD = 0.3  # Below this → route to big even if router says local
LOCAL_CONFIDENCE_THRESHOLD = 0.7   # Below this local confidence → escalate to big
VERIFICATION_FAIL_THRESHOLD = 0.5  # Verifier confidence below this → treat as fail

print(f"Local model: {LOCAL_MODEL}")
providers = []
if USE_FIREWORKS:
    providers.append(f"Fireworks ({FIREWORKS_MODEL})")
if USE_MISTRAL:
    providers.append(f"Mistral ({MISTRAL_MODEL})")
if not USE_FIREWORKS and not USE_MISTRAL:
    providers.append(f"vLLM local ({BIG_MODEL_LOCAL})")
print(f"Big solver:  {' -> '.join(providers) if providers else 'vLLM (' + BIG_MODEL_LOCAL + ')'}")
print(f"Input path:  {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Fireworks key: {bool(FIREWORKS_API_KEY)} | Mistral key: {bool(MISTRAL_API_KEY)}")

In [ ]:
# ============================================================
# Cell 4: vLLM Engine Initialization
# ============================================================

print("Initializing local vLLM engine (small model)...")
t0 = time.time()

llm_local = LLM(
    model=LOCAL_MODEL,
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.25,
    seed=42
)

print(f"  Loaded in {time.time() - t0:.1f}s")

# Initialize big model engine only if no API provider is configured
llm_big = None
if not USE_FIREWORKS and not USE_MISTRAL:
    print("Initializing big vLLM engine (local fallback)...")
    t0 = time.time()
    llm_big = LLM(
        model=BIG_MODEL_LOCAL,
        dtype="half",
        max_model_len=4096,
        gpu_memory_utilization=0.35,
        seed=42
    )
    print(f"  Loaded in {time.time() - t0:.1f}s")
else:
    print("Big solver: API provider(s) configured (no big vLLM engine needed)")

In [ ]:
# ============================================================
# Cell 5: API Clients & Tokenizer
# ============================================================

fw_client = None
if USE_FIREWORKS:
    fw_client = OpenAI(
        api_key=FIREWORKS_API_KEY,
        base_url=FIREWORKS_BASE_URL
    )
    print("Fireworks client initialized.")

mistral_client = None
if USE_MISTRAL:
    mistral_client = OpenAI(
        api_key=MISTRAL_API_KEY,
        base_url=MISTRAL_BASE_URL
    )
    print("Mistral client initialized.")

if not USE_FIREWORKS and not USE_MISTRAL:
    print("No API clients: using vLLM local for all inference.")

# Get tokenizer from local model
tokenizer = llm_local.get_tokenizer()
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")

In [ ]:
# ============================================================
# Cell 6: Helper Functions
# ============================================================

def count_tokens(text: str) -> int:
    """Count tokens in a string using the local model's tokenizer."""
    return len(tokenizer.encode(text))


def format_chat(messages: list) -> str:
    """Apply chat template for vLLM generation."""
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def llm_generate(
    llm: LLM,
    messages: list,
    temperature: float = 0.1,
    max_tokens: int = 1024
) -> str:
    """Generate text using vLLM offline inference."""
    prompt = format_chat(messages)
    params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|endoftext|>", "<|end|>"],
    )
    outputs = llm.generate([prompt], params)
    return outputs[0].outputs[0].text.strip()


def extract_json(text: str) -> Optional[dict]:
    """Extract JSON object from model output."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def parse_confidence(text: str) -> float:
    """Extract a confidence score (0-1) from model output."""
    # Try to find a float between 0 and 1
    matches = re.findall(r'(?<!\d)(0?\.\d+|1\.0)(?!\d)', text)
    for m in matches:
        try:
            val = float(m)
            if 0 <= val <= 1:
                return val
        except ValueError:
            continue
    return 0.5


print("Helper functions loaded.")
import functools
import time as _time

def retry_on_failure(max_retries=3, delay=1.0):
    """Decorator: retry API call on failure with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    if attempt < max_retries - 1:
                        _time.sleep(delay * (2 ** attempt))
            raise last_exc
        return wrapper
    return decorator


In [ ]:
# ============================================================
# Cell 7: Router — Task Type Classification + Few-Shot Routing
# ============================================================

TYPE_CLASSIFICATION_PROMPT = """Classify the task type. Reply with exactly one word: math, code, reasoning, factual, or creative.

Task: {task}
Type:"""

ROUTER_PROMPT = """You are a routing agent that decides which model should handle a task.

Rules:
- LOCAL: simple math, factual questions, straightforward reasoning, well-known facts
- BIG: complex reasoning, code generation, multi-step analysis, creative writing, nuanced comparisons

Examples:
Task: What is 15% of 200?
Type: math
Route: LOCAL
Confidence: 0.9

Task: Write a Python function to merge two sorted lists
Type: code
Route: BIG
Confidence: 0.8

Task: What is the capital of France?
Type: factual
Route: LOCAL
Confidence: 1.0

Task: Compare the advantages of Rust vs Go for systems programming
Type: reasoning
Route: BIG
Confidence: 0.9

Task: Explain the theory of relativity in simple terms
Type: factual
Route: LOCAL
Confidence: 0.8

Task: Debug this code: def f(x): return x / 0
Type: code
Route: BIG
Confidence: 0.9
"""

# We'll use a simpler two-step approach internally

def route_task(task: str) -> dict:
    """
    Classify task type and decide route.
    Returns: {"task_type": str, "router_decision": "local"|"big", "confidence": float}
    """
    # Step 1: Classify task type
    type_messages = [
        {"role": "system", "content": "You classify tasks into types. Reply with one word only."},
        {"role": "user", "content": TYPE_CLASSIFICATION_PROMPT.format(task=task)}
    ]
    task_type = llm_generate(llm_local, type_messages, temperature=0.1).strip().lower()

    # Normalize task type
    valid_types = ["math", "code", "reasoning", "factual", "creative"]
    for t in valid_types:
        if t in task_type:
            task_type = t
            break
    else:
        task_type = "reasoning"  # default

    # Step 2: Routing decision + confidence
    route_messages = [
        {"role": "system", "content": ROUTER_PROMPT},
        {"role": "user", "content": f"Task: {task}\nType: {task_type}\nRoute:"}
    ]
    decision_raw = llm_generate(llm_local, route_messages, temperature=0.1).strip()

    # Parse decision
    decision_upper = decision_raw.upper()
    if "BIG" in decision_upper and "LOCAL" not in decision_upper:
        is_local = False
    elif "LOCAL" in decision_upper:
        is_local = True
    else:
        # Fallback based on task type
        is_local = task_type in ("math", "factual")

    # Extract confidence from the decision output
    confidence = parse_confidence(decision_raw)

    # Apply confidence threshold: low confidence → escalate to big
    if is_local and confidence < ROUTER_CONFIDENCE_THRESHOLD:
        is_local = False

    return {
        "task_type": task_type,
        "router_decision": "local" if is_local else "big",
        "confidence": confidence,
        "raw_output": decision_raw[:100]
    }


# Quick test
if os.getenv("RUN_TESTS", "0") == "1":    print("Router test:", route_task("What is 2+2?"))

In [ ]:
# ============================================================
# Cell 8: Local Solver (vLLM Qwen2.5-3B)
# ============================================================

LOCAL_SOLVE_PROMPT = """Solve the following task. Think step by step, then provide your answer.

Task: {task}

Respond with valid JSON only:
{{
  "thinking": "your step-by-step reasoning",
  "answer": "your final concise answer",
  "confidence": 0.95
}}"""


def solve_local(task: str) -> dict:
    """Solve a task using the local small model. Returns dict with thinking, answer, confidence."""
    messages = [
        {"role": "system", "content": "You are a precise AI assistant. Always respond with valid JSON."},
        {"role": "user", "content": LOCAL_SOLVE_PROMPT.format(task=task)}
    ]
    raw = llm_generate(llm_local, messages, temperature=0.1)

    result = extract_json(raw)
    if result and "answer" in result:
        result["raw_tokens"] = count_tokens(raw)
        return result

    # Fallback: return raw text
    return {
        "thinking": "",
        "answer": raw,
        "confidence": 0.3,
        "raw_tokens": count_tokens(raw),
        "parse_failed": True
    }


# Quick test
test = solve_local("What is 25 * 4 + 10?")
print(f"Answer: {test.get('answer', 'N/A')}")
print(f"Confidence: {test.get('confidence', 0)}")

In [ ]:
# ============================================================
# Cell 9: Big Solver — Fireworks / Mistral / vLLM (fallback chain)
# ============================================================

BIG_SOLVE_SYSTEM_PROMPT = "You are a precise AI assistant. Respond with valid JSON containing 'thinking', 'answer', and 'confidence' fields."

BIG_SOLVE_USER_PROMPT = """Task: {task}

Respond with JSON:
{{
  "thinking": "your step-by-step reasoning",
  "answer": "your final concise answer",
  "confidence": 0.95
}}"""


@retry_on_failure(max_retries=3, delay=1.0)
def solve_big_fireworks(task: str) -> dict:
    """Solve using Fireworks AI API (72B model). Tracks token usage."""
    response = fw_client.chat.completions.create(
        model=FIREWORKS_MODEL,
        messages=[
            {"role": "system", "content": BIG_SOLVE_SYSTEM_PROMPT},
            {"role": "user", "content": BIG_SOLVE_USER_PROMPT.format(task=task)}
        ],
        temperature=0.3,
        max_tokens=4096,
        response_format={"type": "json_object"}
    )
    content = response.choices[0].message.content
    usage = response.usage

    result = json.loads(content)
    result["api_tokens"] = {
        "total": usage.total_tokens,
        "prompt": usage.prompt_tokens,
        "completion": usage.completion_tokens,
        "provider": "fireworks"
    }
    return result


@retry_on_failure(max_retries=3, delay=1.0)
def solve_big_mistral(task: str) -> dict:
    """Solve using Mistral API (Mistral Large). Tracks token usage."""
    # Mistral API is OpenAI-compatible but does not support response_format
    response = mistral_client.chat.completions.create(
        model=MISTRAL_MODEL,
        messages=[
            {"role": "system", "content": BIG_SOLVE_SYSTEM_PROMPT + " Return ONLY valid JSON."},
            {"role": "user", "content": BIG_SOLVE_USER_PROMPT.format(task=task)}
        ],
        temperature=0.3,
        max_tokens=4096
    )
    content = response.choices[0].message.content
    usage = response.usage

    result = extract_json(content) or json.loads(content)
    result["api_tokens"] = {
        "total": usage.total_tokens,
        "prompt": usage.prompt_tokens,
        "completion": usage.completion_tokens,
        "provider": "mistral"
    }
    return result


def solve_big_vllm(task: str) -> dict:
    """Solve using local larger vLLM model (14B). Free inference, no token tracking."""
    messages = [
        {"role": "system", "content": BIG_SOLVE_SYSTEM_PROMPT},
        {"role": "user", "content": BIG_SOLVE_USER_PROMPT.format(task=task)}
    ]
    raw = llm_generate(llm_big, messages, temperature=0.3)
    result = extract_json(raw)
    if result and "answer" in result:
        result["api_tokens"] = {"total": 0, "prompt": 0, "completion": 0, "provider": "vllm"}
        return result
    return {
        "thinking": "",
        "answer": raw,
        "confidence": 0.5,
        "api_tokens": {"total": 0, "prompt": 0, "completion": 0, "provider": "vllm"}
    }


def solve_big(task: str) -> dict:
    """Fallback chain: Fireworks -> Mistral -> vLLM local."""
    if USE_FIREWORKS:
        try:
            return solve_big_fireworks(task)
        except Exception as e:
            print(f"Fireworks failed ({e}), falling back to Mistral...")
    if USE_MISTRAL:
        try:
            return solve_big_mistral(task)
        except Exception as e:
            print(f"Mistral failed ({e}), falling back to vLLM local...")
    return solve_big_vllm(task)


print(f"Big solver: Fireworks={USE_FIREWORKS} Mistral={USE_MISTRAL} vLLM={'always available'}")

In [ ]:
# ============================================================
# Cell 10: Verifier — Re-Answer from Scratch
# ============================================================

VERIFY_PROMPT = """Solve this task from scratch. Do NOT reference any previous answer.

Task: {task}

Think step by step and respond with valid JSON:
{{
  "thinking": "your reasoning",
  "answer": "your answer",
  "confidence": 0.95
}}"""


def verify_answer(task: str, original_answer: dict) -> dict:
    """
    Re-answer the task from scratch using the local model.
    Compares the new answer to the original answer.
    Returns dict with passed (bool), answers, and reason.
    """
    messages = [
        {"role": "system", "content": "Solve the task from scratch. Do NOT reference existing answers."},
        {"role": "user", "content": VERIFY_PROMPT.format(task=task)}
    ]
    raw = llm_generate(llm_local, messages, temperature=0.1)
    verified = extract_json(raw)

    if verified is None:
        return {
            "passed": True,  # Default: trust original
            "reason": "Could not parse verification output",
            "original_answer": str(original_answer.get("answer", "")),
            "verified_answer": raw[:100],
            "verifier_confidence": 0.0
        }

    orig_ans = str(original_answer.get("answer", "")).strip().lower()
    verif_ans = str(verified.get("answer", "")).strip().lower()
    verif_conf = verified.get("confidence", 0.5)

    # Normalize both answers: try numeric comparison first, fall back to string
    try:
        orig_num = float(orig_ans.replace(',', '').replace('$', '').replace('%', '').strip())
        verif_num = float(verif_ans.replace(',', '').replace('$', '').replace('%', '').strip())
        passed = abs(orig_num - verif_num) < 0.01
    except ValueError:
        passed = orig_ans == verif_ans or (orig_ans in verif_ans and len(orig_ans) > 3) or (verif_ans in orig_ans and len(verif_ans) > 3)

    # If verifier has low confidence, treat as potential failure
    if verif_conf < VERIFICATION_FAIL_THRESHOLD:
        passed = False

    return {
        "passed": passed,
        "reason": "Match" if passed else f"Mismatch | orig: '{orig_ans[:50]}' vs verif: '{verif_ans[:50]}'",
        "original_answer": orig_ans,
        "verified_answer": verif_ans,
        "verifier_confidence": verif_conf
    }


print("Verifier loaded.")

In [ ]:
# ============================================================
# Cell 11: Orchestration Pipeline
# ============================================================

RETRY_PROMPT = """Think step by step, carefully. Solve this task.

Task: {task}

Provide JSON:
{{
  "thinking": "detailed step-by-step",
  "answer": "final answer",
  "confidence": 0.95
}}"""


def process_task(task_id: Any, task_text: str) -> dict:
    """
    Full pipeline for a single task:
      1. Route (type classification + decision + confidence)
      2. Solve (local or big based on routing)
      3. Verify (re-answer from scratch)
      4. Retry local once if verification fails
      5. Escalate to big if retry also fails
    Returns result dict with all metadata.
    """
    result = {
        "id": task_id,
        "task": task_text,
        "answer": "",
        "thinking": "",
        "confidence": 0.0,
        "api_tokens": {"total": 0, "provider": ""},
        "routing": {},
        "verification": {},
        "retries": 0,
        "escalated": False
    }

    # Step 1: Route
    route_info = route_task(task_text)
    result["routing"] = route_info

    # Step 2: Solve
    if route_info["router_decision"] == "local":
        # Try local solver
        answer = solve_local(task_text)
        result["answer"] = answer.get("answer", "")
        result["thinking"] = answer.get("thinking", "")
        result["confidence"] = answer.get("confidence", 0.0)

        # Check confidence threshold
        if result["confidence"] < LOCAL_CONFIDENCE_THRESHOLD:
            # Low confidence → go straight to big solver
            route_info["low_confidence_override"] = True
            big_answer = solve_big(task_text)
            result["answer"] = big_answer.get("answer", "")
            result["thinking"] = big_answer.get("thinking", "")
            result["confidence"] = big_answer.get("confidence", 0.0)
            result["api_tokens"] = big_answer.get("api_tokens", {"total": 0, "provider": "big"})
            result["escalated"] = True
        else:
            # Step 3: Verify with re-answer
            verification = verify_answer(task_text, answer)
            result["verification"] = verification

            if not verification["passed"]:
                # Step 4: Retry local once with CoT
                result["retries"] = 1
                retry_messages = [
                    {"role": "system", "content": "You solve tasks carefully step by step."},
                    {"role": "user", "content": RETRY_PROMPT.format(task=task_text)}
                ]
                retry_raw = llm_generate(llm_local, retry_messages, temperature=0.1)
                retry_answer = extract_json(retry_raw)

                if retry_answer and retry_answer.get("confidence", 0) > result["confidence"]:
                    # Retry produced a more confident answer
                    answer = retry_answer
                    result["answer"] = answer.get("answer", "")
                    result["thinking"] = answer.get("thinking", "")
                    result["confidence"] = answer.get("confidence", 0.0)

                    # Re-verify
                    verification = verify_answer(task_text, answer)
                    result["verification"] = verification

                if not verification["passed"]:
                    # Step 5: Escalate to big
                    route_info["escalated"] = True
                    result["escalated"] = True
                    big_answer = solve_big(task_text)
                    result["answer"] = big_answer.get("answer", "")
                    result["thinking"] = big_answer.get("thinking", "")
                    result["confidence"] = big_answer.get("confidence", 0.0)
                    result["api_tokens"] = big_answer.get("api_tokens", {"total": 0, "provider": "big"})
    else:
        # Route directly to big
        big_answer = solve_big(task_text)
        result["answer"] = big_answer.get("answer", "")
        result["thinking"] = big_answer.get("thinking", "")
        result["confidence"] = big_answer.get("confidence", 0.0)
        result["api_tokens"] = big_answer.get("api_tokens", {"total": 0, "provider": "big"})
        result["escalated"] = True

    return result


def run_pipeline(tasks: list) -> list:
    """Run the full pipeline on a list of task dicts."""
    results = []
    total_api_tokens = 0
    local_count = 0
    big_count = 0

    for i, task_item in enumerate(tasks):
        task_id = task_item.get("id", i)
        task_text = task_item.get("task", task_item.get("question", str(task_item)))
        print(f"[{i+1}/{len(tasks)}] task {task_id}: {task_text[:60]}...")
        t0 = time.time()

        result = process_task(task_id, task_text)
        elapsed = time.time() - t0

        result["elapsed_seconds"] = round(elapsed, 2)
        results.append(result)
        total_api_tokens += result["api_tokens"]["total"]

        route = result["routing"].get("router_decision", "?")
        if result["escalated"]:
            route += " → big"
            big_count += 1
        else:
            local_count += 1

        prov = result["api_tokens"].get("provider", "?")
        tok = result["api_tokens"]["total"]
        print(f"  Route: {route} | Conf: {result['confidence']:.2f} | {prov} tok: {tok} | {elapsed:.1f}s")
        print(f"  Answer: {str(result['answer'])[:80]}")

    print(f"\n=== Summary ===")
    print(f"Total tasks:      {len(tasks)}")
    print(f"Local solved:     {local_count} ({local_count/len(tasks)*100:.0f}%)")
    print(f"Big/escalated:    {big_count} ({big_count/len(tasks)*100:.0f}%)")
    print(f"API tokens:       {total_api_tokens}")
    print(f"Avg tokens/task:  {total_api_tokens/len(tasks):.1f}")

    return results


print("Pipeline ready.")

In [ ]:
# ============================================================
# Cell 12: Data Loading
# ============================================================

def load_tasks(path: str = None) -> list:
    """Load tasks from JSON file. Returns list of dicts with at least 'task' key."""
    if path is None:
        path = INPUT_PATH

    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f"Loaded {len(data)} tasks from {path}")
            return data
        elif isinstance(data, dict):
            # Might be wrapped in a key
            for key in ["tasks", "data", "items", "questions"]:
                if key in data and isinstance(data[key], list):
                    print(f"Loaded {len(data[key])} tasks from {path}.{key}")
                    return data[key]
            return [data]  # Single task
    else:
        print(f"Input path {path} not found. Using sample tasks instead.")
        return generate_sample_tasks()


def generate_sample_tasks() -> list:
    """Generate a diverse set of sample tasks for testing."""
    return [
        {"id": 1, "task": "What is 15% of 200?"},
        {"id": 2, "task": "Write a Python function to check if a string is a palindrome"},
        {"id": 3, "task": "What is the capital of Japan?"},
        {"id": 4, "task": "Explain the difference between TCP and UDP protocols"},
        {"id": 5, "task": "Write a sonnet about artificial intelligence"},
        {"id": 6, "task": "If a train travels at 60 mph for 2.5 hours, how far does it go?"},
        {"id": 7, "task": "Debug this: why does 'print(1/0)' crash in Python?"},
        {"id": 8, "task": "Compare the advantages of Rust vs Go for systems programming"},
        {"id": 9, "task": "What is the chemical formula for water?"},
        {"id": 10, "task": "Create a meal plan for a vegetarian athlete"},
    ]


print("Data loader ready.")

In [ ]:
# ============================================================
# Cell 13: Run Pipeline
# ============================================================

# Load tasks
tasks = load_tasks()
print(f"Tasks to process: {len(tasks)}\n")

# Run the pipeline
results = run_pipeline(tasks)

# Optionally write results
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults written to {OUTPUT_PATH}")

In [ ]:
# ============================================================
# Cell 14: Results Analysis
# ============================================================

def analyze_results(results: list):
    """Print detailed analysis of pipeline results."""
    api_tokens_total = sum(r.get("api_tokens", {}).get("total", 0) for r in results)
    local_only = sum(1 for r in results if not r.get("escalated", False))
    escalated = sum(1 for r in results if r.get("escalated", False))
    avg_confidence = sum(r.get("confidence", 0) for r in results) / len(results)
    total_time = sum(r.get("elapsed_seconds", 0) for r in results)

    print("=" * 50)
    print("TOKEN SAGE  —  RESULTS ANALYSIS")
    print("=" * 50)
    print(f"{'Total tasks':30s} {len(results)}")
    print(f"{'Solved locally (free)':30s} {local_only} ({local_only/len(results)*100:.1f}%)")
    print(f"{'Escalated to big':30s} {escalated} ({escalated/len(results)*100:.1f}%)")
    print(f"{'Avg confidence':30s} {avg_confidence:.2f}")
    print(f"{'Total API tokens':30s} {api_tokens_total}")
    print(f"{'Avg API tokens/task':30s} {api_tokens_total/len(results):.1f}")
    print(f"{'Total processing time':30s} {total_time:.1f}s")

    if api_tokens_total > 0:
        print(f"\n{'Token efficiency (tasks per API token)':30s} {len(results)/api_tokens_total:.4f}")

    print("\n" + "=" * 50)
    print("PER-TASK BREAKDOWN")
    print("=" * 50)
    for r in results:
        route = r["routing"].get("router_decision", "?")
        if r.get("escalated"):
            route += " → big"
        ans_preview = str(r.get("answer", ""))[:50]
        print(f"  #{r['id']:<4} [{route:<15}] conf={r['confidence']:.2f}  tok={r.get('api_tokens',{}).get('total',0):>5}  {r.get('elapsed_seconds',0):.1f}s  {ans_preview}")


analyze_results(results)

---
## Next Steps

1. **Tweak routing prompts** — adjust few-shot examples in `ROUTER_PROMPT` (Cell 7) to improve routing accuracy
2. **Adjust thresholds** — modify `ROUTER_CONFIDENCE_THRESHOLD` and `LOCAL_CONFIDENCE_THRESHOLD` (Cell 3)
3. **Test with scoring data** — place actual tasks at `/input/tasks.json` and re-run Cell 13
4. **Extract to production** — when ready, copy core logic to `agent/main.py` and build `Dockerfile`
5. **Consider DistilBERT upgrade** — if LLM routing cost is too high, fine-tune a DistilBERT classifier for zero-token routing